<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Módulo 2</h2><br/>
<h1>Semana 6 · Miércoles — Random Forest y Bagging</h1>
<h3>El poder de combinar muchos árboles</h3>
<br/>
    <b>Instructor:</b> Jesús Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## 🎯 Objetivos de hoy

Al final vas a poder:

1. Entender el concepto de **ensemble learning** y por qué "muchos modelos juntos" suelen ganar.
2. Entender **Bagging** (Bootstrap Aggregating) y cómo reduce el overfitting.
3. Entender **Random Forest** como una versión mejorada de Bagging.
4. Dominar los parámetros clave: `n_estimators`, `max_depth`, `max_features`.
5. Extraer e interpretar la **Feature Importance**.
6. Resolver **3 ejercicios** comparando árbol simple vs random forest.

# 1. Ensemble Learning — la sabiduría de las multitudes

### 🧠 Analogía: el examen grupal

Imagina un examen de selección múltiple. Tienes dos opciones:

- 🧑 **Hacerlo solo**: tu respuesta es lo que sabes — si te equivocas, te equivocas.
- 👥 **Hacerlo en grupo de 100 personas**: cada uno responde, y la respuesta final es **la votación mayoritaria**. Incluso si cada persona individual no es genio, el grupo casi siempre acierta más.

Eso es **Ensemble Learning**: entrenar MUCHOS modelos y combinar sus predicciones para obtener un resultado **más robusto y preciso** que cualquier modelo individual.

### Tipos principales

| Tipo | Idea | Ejemplo |
|---|---|---|
| **Bagging** | Entrenar muchos modelos sobre **muestras aleatorias** del dataset y promediar | Random Forest |
| **Boosting** | Entrenar modelos en **secuencia**, cada uno corrige al anterior | XGBoost, LightGBM |
| **Stacking** | Entrenar modelos diferentes y un meta-modelo que aprende a combinarlos | Combinaciones avanzadas |

Hoy nos enfocamos en **Bagging** y **Random Forest**. Boosting viene en semana 7.

# 2. Bagging — Bootstrap Aggregating

**Bagging** = **B**ootstrap **Agg**regat**ing**. La idea:

1. Tomar **N muestras aleatorias con reemplazo** del dataset (bootstrap).
2. Entrenar **un modelo (típicamente un árbol) sobre cada muestra**.
3. Para predecir un caso nuevo: **promediar las predicciones** de todos los modelos.

### ¿Por qué funciona?

Los árboles individuales tienden a **sobreajustar** (memorizan el train). Si entrenas 100 árboles distintos sobre subconjuntos distintos, cada uno se equivoca en cosas distintas → al promediar, los errores **se cancelan entre sí**.

### 🧠 Analogía

Es como pedirle a 100 amigos diferentes que estimen tu peso. Cada uno tiene un sesgo (uno cree pesas más, otro menos), pero el promedio se acerca mucho a la verdad.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Dataset desafiante
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1) Un solo árbol — modelo base
arbol_simple = DecisionTreeRegressor(max_depth=10, random_state=42).fit(X_train, y_train)
print(f'1 Árbol solo:        R² = {arbol_simple.score(X_test, y_test):.4f}')

# 2) Bagging con 100 árboles
bagging = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=10, random_state=42),
    n_estimators=100,
    random_state=42,
    n_jobs=-1
).fit(X_train, y_train)
print(f'Bagging con 100:     R² = {bagging.score(X_test, y_test):.4f}')
print('\n👉 Bagging mejora notablemente el R² simplemente combinando árboles.')

# 3. Random Forest — Bagging mejorado

**Random Forest** es Bagging + un truco adicional: en cada split del árbol, **considera solo un subconjunto aleatorio de features**.

### ¿Por qué este truco?

Si todos los árboles ven todas las features, tienden a usar la **misma feature dominante** en cada split → son todos parecidos. Al limitar las features por split, fuerzas **diversidad** entre los árboles.

Más diversidad = mejor promedio = mejor modelo.

## Parámetros clave

| Parámetro | ¿Qué hace? | Típico |
|---|---|---|
| `n_estimators` | Cuántos árboles entrenar | 100, 200, 500 |
| `max_depth` | Profundidad máxima de cada árbol | 10-20 o `None` |
| `max_features` | Features consideradas por split | `'sqrt'` para clasificación, `1.0` para regresión |
| `min_samples_leaf` | Mínimo de muestras por hoja | 1-5 |
| `n_jobs` | Paralelización (`-1` usa todos los cores) | `-1` (¡rápido!) |
| `random_state` | Reproducibilidad | 42 |

In [ ]:
# Random Forest con 200 árboles
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    n_jobs=-1,
    random_state=42
).fit(X_train, y_train)

print(f'Random Forest:       R² = {rf.score(X_test, y_test):.4f}')
print(f'MAE:                 ${mean_absolute_error(y_test, rf.predict(X_test))*100_000:.0f} USD')
print(f'\n💡 Comparación con los anteriores:')
print(f'   1 árbol:   R² ≈ {arbol_simple.score(X_test, y_test):.3f}')
print(f'   Bagging:   R² ≈ {bagging.score(X_test, y_test):.3f}')
print(f'   RForest:   R² ≈ {rf.score(X_test, y_test):.3f}')

In [ ]:
# ¿Cuántos árboles necesitamos? — curva del rendimiento
n_arboles = [1, 5, 10, 25, 50, 100, 200, 500]
scores = []

for n in n_arboles:
    m = RandomForestRegressor(n_estimators=n, max_depth=15, n_jobs=-1, random_state=42)
    m.fit(X_train, y_train)
    scores.append(m.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(n_arboles, scores, marker='o', linewidth=2, color='#70AD47')
ax.set_xlabel('n_estimators (número de árboles)')
ax.set_ylabel('R² en test')
ax.set_title('Rendimiento de Random Forest según cantidad de árboles')
ax.set_xscale('log')
ax.grid(alpha=0.3)
plt.show()

print('👉 El R² mejora rápido al principio y se estabiliza a partir de ~100 árboles.')
print('   Pasar de 100 a 500 árboles aumenta el tiempo de entrenamiento sin gran mejora.')

# 4. Feature Importance — la joya del Random Forest

Random Forest tiene una **superpoder**: te dice **qué variables son más importantes** para predecir.

Esto es ENORME para:
- Explicar tu modelo al negocio ("el ingreso es la variable #1")
- Reducir features irrelevantes
- Generar insights del dominio

Cada árbol del bosque calcula cuánto **reduce el error** cada feature. Random Forest **promedia esa importancia** entre todos los árboles.

In [ ]:
# Extraer feature importance
importancia = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(importancia.index, importancia.values, color='steelblue', edgecolor='white')
for i, v in enumerate(importancia.values):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontweight='bold')
ax.set_xlabel('Importancia')
ax.set_title('Feature Importance — Random Forest sobre California Housing', fontweight='bold')
ax.set_xlim(0, importancia.max() * 1.15)
plt.tight_layout()
plt.show()

print('Top 3 variables más importantes:')
print(importancia.sort_values(ascending=False).head(3))
print('\n👉 MedInc (ingreso medio) es de lejos la variable más predictiva del precio.')

# 5. Comparación final — ¿qué ganó hoy?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

modelos = {
    'Regresión Lineal':       make_pipeline(StandardScaler(), LinearRegression()),
    'Árbol único (depth=10)': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Bagging (100 árboles)':  BaggingRegressor(DecisionTreeRegressor(max_depth=10), n_estimators=100, n_jobs=-1, random_state=42),
    'Random Forest (200)':    RandomForestRegressor(n_estimators=200, max_depth=15, n_jobs=-1, random_state=42),
}

resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados.append({
        'Modelo': nombre,
        'R²':  r2_score(y_test, pred),
        'MAE': mean_absolute_error(y_test, pred)
    })

tabla = pd.DataFrame(resultados).set_index('Modelo').round(4)
print(tabla)

fig, ax = plt.subplots(figsize=(10, 4))
tabla['R²'].plot(kind='barh', color=['#4472C4', '#ED7D31', '#A0A0A0', '#70AD47'], edgecolor='white', ax=ax)
for i, v in enumerate(tabla['R²']):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontweight='bold')
ax.set_xlabel('R²'); ax.set_title('Random Forest gana claramente', fontweight='bold')
plt.tight_layout(); plt.show()

---
# 🏋️ Ejercicios prácticos

## Ejercicio 1 — Árbol vs Random Forest sobre `diamonds`

**Tarea:**

1. Cargar `sns.load_dataset('diamonds')`.
2. Codificar las 3 ordinales (`cut`, `color`, `clarity`) con `OrdinalEncoder` respetando el orden (como ayer en KNN).
3. Target: `price`. Features: el resto.
4. Train/test 80/20.
5. Entrenar **2 modelos**:
   - Un `DecisionTreeRegressor(max_depth=10, random_state=42)`
   - Un `RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)`
6. Comparar R² y MAE.
7. ¿Cuánto mejor es el RF? Comenta.

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

df = sns.load_dataset('diamonds').copy()
enc = OrdinalEncoder(categories=[
    ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal'],
    ['J', 'I', 'H', 'G', 'F', 'E', 'D'],
    ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']
])
df[['cut', 'color', 'clarity']] = enc.fit_transform(df[['cut', 'color', 'clarity']])

X = df.drop(columns=['price'])
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

arbol = DecisionTreeRegressor(max_depth=10, random_state=42).fit(X_train, y_train)
rf    = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42).fit(X_train, y_train)

for nombre, m in [('Árbol', arbol), ('Random Forest', rf)]:
    pred = m.predict(X_test)
    print(f'{nombre:15s} → R²={r2_score(y_test, pred):.4f}, MAE=${mean_absolute_error(y_test, pred):.0f}')
# 👉 RF mejora ~3-5% el R² y reduce notablemente el MAE.
```
</details>

## Ejercicio 2 — Feature Importance con `taxis`

**Tarea:**

1. Cargar `sns.load_dataset('taxis').dropna()`.
2. Target: `total` (precio total del viaje).
3. Features útiles: `distance`, `passengers`, `pickup_borough`, `dropoff_borough`, `payment`, `color` (NO uses `fare/tip/tolls`).
4. Codificar categóricas con `pd.get_dummies(... drop_first=True)`.
5. Train/test 80/20.
6. Entrenar `RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)`.
7. Sacar y graficar la **Feature Importance** ordenada.
8. ¿Cuál es la variable #1? ¿Tiene sentido?

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

taxis = sns.load_dataset('taxis').dropna()
features = ['distance', 'passengers', 'pickup_borough', 'dropoff_borough', 'payment', 'color']
df = pd.get_dummies(taxis[features + ['total']], drop_first=True, dtype=int)

X = df.drop(columns=['total'])
y = df['total']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42).fit(X_train, y_train)
print(f'R² test: {rf.score(X_test, y_test):.4f}')

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
imp.plot(kind='barh', color='steelblue', edgecolor='white', ax=ax)
ax.set_title('Feature Importance — Predicción de Total de Taxi NYC')
plt.tight_layout(); plt.show()

# 👉 'distance' suele ser la #1 con ~85% de importancia.
#    A mayor distancia, mayor tarifa — TOTALMENTE intuitivo.
```
</details>

## Ejercicio 3 — Tuning de Random Forest

**Tarea:** Encontrar la mejor combinación de hiperparámetros para predecir `body_mass_g` de `penguins`.

1. Cargar `sns.load_dataset('penguins').dropna()`.
2. Aplicar `pd.get_dummies()` a las categóricas.
3. Train/test 80/20.
4. Hacer un **doble loop** sobre:
   - `n_estimators = [50, 100, 200, 500]`
   - `max_depth = [5, 10, 15, None]`
5. Guardar el R² de cada combinación en un DataFrame.
6. **Mostrarlo como heatmap** (`sns.heatmap`).
7. ¿Cuál es la mejor combinación?

**Pista:** este es un **tuning manual**. La semana próxima veremos `GridSearchCV` que automatiza esto.

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

pen = sns.load_dataset('penguins').dropna()
df = pd.get_dummies(pen, columns=['species', 'island', 'sex'], drop_first=True, dtype=int)
X = df.drop(columns=['body_mass_g'])
y = df['body_mass_g']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

n_arboles = [50, 100, 200, 500]
depths    = [5, 10, 15, None]

tabla = pd.DataFrame(index=n_arboles, columns=depths)
for n in n_arboles:
    for d in depths:
        m = RandomForestRegressor(n_estimators=n, max_depth=d, n_jobs=-1, random_state=42)
        m.fit(X_train, y_train)
        tabla.loc[n, d] = round(m.score(X_test, y_test), 4)

print(tabla)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(tabla.astype(float), annot=True, fmt='.4f', cmap='viridis', ax=ax)
ax.set_xlabel('max_depth'); ax.set_ylabel('n_estimators')
ax.set_title('R² en test — tuning de Random Forest')
plt.show()

# 👉 Suele ganar n_estimators alto con max_depth moderado (10-15).
```
</details>

---
## 📌 Cierre del día

Hoy aprendimos:

- ✅ **Ensemble Learning** — muchos modelos juntos > un solo modelo
- ✅ **Bagging** — entrenar árboles sobre muestras aleatorias y promediar
- ✅ **Random Forest** — Bagging + selección aleatoria de features por split
- ✅ Los parámetros clave: `n_estimators`, `max_depth`, `max_features`, `n_jobs`
- ✅ **Feature Importance** — la joya explicable de RF
- ✅ Random Forest casi siempre supera a un árbol único

### 🔜 Mañana — Jueves 28

- **Benchmarking sistemático** — comparar 5+ modelos a la vez
- **LazyPredict** — autobenchmark de docenas de modelos en una línea
- Optimización inicial de hiperparámetros con GridSearch

Nos vemos 🚀